# 04 - Differential Abundance (TP1)

**Paper Title:** Maternal gut microbiome diversity and functional potential in early pregnancy are associated with large-for-gestational-age birth (SweMaMi cohort)

**Purpose:** Runs MaAsLin3 differential abundance and prevalence analysis at TP1, with birth weight category as the fixed effect of interest, adjusting for confounders and read depth. Significance at qval_individual ≤ 0.1.

**Produces:** Figure 5 (differential species), Supplementary Data 3


## Setup


In [ ]:
library(maaslin3)
library(dplyr)
library(data.table)
library(tibble)
library(ggplot2)
library(tidyverse)

## 1. Data preparation


In [ ]:
# Define base path 
base_path <- "data"

In [ ]:
meta_data_tp1 <- read.csv(file.path(base_path, "meta_data_tp1.csv"), 
                           header = TRUE)
meta_data_tp1$Primipara <- factor(meta_data_tp1$Primipara, levels = c("0","1"))
meta_data_tp1$excess_weight <- factor(meta_data_tp1$excess_weight, levels = c("No","Yes"))
meta_data_tp1$Gest_Diabetes <- factor(meta_data_tp1$Gest_Diabetes,levels = c("0","1"))
meta_data_tp1$Q1_healthydiet <- factor(meta_data_tp1$Q1_healthydiet, levels = c("1","0"))
meta_data_tp1$Q2_healthydiet <- factor(meta_data_tp1$Q2_healthydiet, levels = c("1","0"))
meta_data_tp1$Group <- relevel(factor(meta_data_tp1$Group), ref = "Control")


In [ ]:
row.names(meta_data_tp1) <- meta_data_tp1$kit1.faecal_sample.barcode
meta_data_tp1 <- meta_data_tp1[ , -1]

In [ ]:
all_taxa <- read.table(file.path(base_path, "taxa_tp1.tsv"),
                       sep = "\t", header = TRUE, check.names = FALSE)


In [ ]:
new_row <- c()
tot <- colSums(all_taxa)
for(i in tot){
    if(i < 100) {
        new_row <- c(new_row,100 - i)
    } else if( i >=100){
        new_row <- c(new_row,0)
    }
}

new_row <- setNames(new_row, names(all_taxa))
new_row2 <- as.data.frame(t(new_row))
rownames(new_row2) <- "Other_taxa"
best_filt_new <- rbind(all_taxa,new_row2) 
filt_new_arrange_1 <- as.data.frame(best_filt_new)
dim(filt_new_arrange_1)


In [ ]:
beta_taxa_1 <- filt_new_arrange_1 [,colnames(filt_new_arrange_1) %in% row.names(meta_data_tp1)]
for_arrange_1 <- as.data.frame(t(beta_taxa_1))

In [ ]:
folder_path_1 <- "read_count_files"

files_1 <- list.files(folder_path_1, pattern = "\\.txt$", full.names = TRUE)

read_depth <- files_1 %>%
    # tab-separated by default
  lapply(read.delim) %>%   
  bind_rows()

In [ ]:
read_depth_1<- read_depth %>%
  mutate(
    Sample_ID   = sub(".*__(\\d+)__.*", "\\1", read_depth$Sample),
    Flowcell_ID = sub(".*__(E\\d+).*", "\\1", read_depth$Sample)
  )
read_depth_2 <- read_depth_1 %>% filter(Sample_ID %in% rownames(meta_data_tp1)) %>% 
                                        select(c(Sample_ID, Flowcell_ID, after_kraken2_host_removal))

read_depth_3 <- read_depth_2 %>% rename(read_depth = after_kraken2_host_removal)


In [ ]:
meta_data_tp1 <- meta_data_tp1 %>%
  rownames_to_column(var = "Row.names")

In [ ]:
update_meta_data_tp1 <- full_join(
  meta_data_tp1,
  read_depth_3,
  by = c("Row.names" = "Sample_ID")
)

update_meta_data_tp1 <- update_meta_data_tp1 %>%
  column_to_rownames(var = "Row.names")


for_arrange_11 <- for_arrange_1[rownames(update_meta_data_tp1), ]

## 2. Run MaAsLin3 (abundance + prevalence models)
Heatmap of differential species (Figure 5)

In [ ]:
#Adjusted
Adjust_output_folder_TP1 <- file.path(base_path, "output/TP1_adjust")

In [ ]:
fit_data_TP1_adjust <- maaslin3(
    input_data = for_arrange_11,
    input_metadata = update_meta_data_tp1,
    output = Adjust_output_folder_TP1,
    
    fixed_effects = c('Group', 'age_checked', 'BMI_prior', 
                      'Primipara', 'Q1_healthydiet', 'read_depth'),
    
    reference = "Group,Control;Primipara,0;Q1_healthydiet,1",
    
    normalization = "TSS", 
    transform = "LOG", 
    correction = "BH",
    
   #ADD THESE SPECIFIC PLOTTING ADJUSTMENT: Limits the summary plot to exactly 15 species 
    plot_summary_plot = TRUE, 
    summary_plot_first_n = 15,          
    coef_plot_vars = c("Group Control"),    
    heatmap_vars = c("Group Control"),      
    plot_associations = TRUE, 
    max_pngs = 50,                    
    max_significance = 0.1            
)


## 3. Filter to qval_individual ≤ 0.1


In [ ]:
signi_adjust <- read.table(file.path(base_path, "output/TP1_adjust.tsv"), sep = "\t", header = TRUE)


In [ ]:
# Using qval_individual 
prevalence_sig_correct <- signi_adjust %>%
  filter(metadata == "Group",
         model == "prevalence",
         qval_individual <= 0.1) %>%
  pull(feature)

abundance_sig_correct <- signi_adjust %>%
  filter(metadata == "Group",
         model == "abundance",
         qval_individual <= 0.1) %>%
  pull(feature)

# True classification
both_true  <- intersect(prevalence_sig_correct,
                        abundance_sig_correct)
only_prev  <- setdiff(prevalence_sig_correct,
                      abundance_sig_correct)
only_abund <- setdiff(abundance_sig_correct,
                      prevalence_sig_correct)

cat("Both models:", length(both_true), "\n")
print(both_true)

cat("Prevalence only:", length(only_prev), "\n")
print(only_prev)

cat("Abundance only:", length(only_abund), "\n")
print(only_abund)

In [ ]:
# Direction for prevalence species
prevalence_direction <- signi_adjust %>%
  filter(metadata == "Group",
         model == "prevalence",
         feature %in% only_prev) %>%
  select(feature, coef, qval_individual) %>%
  mutate(
    direction = ifelse(coef > 0, 
                       "More prevalent in LGA", 
                       "More prevalent in AGA"),
    coef = round(coef, 3),
    qval_individual = round(qval_individual, 4)
  ) %>%
  arrange(desc(coef))

cat("=== PREVALENCE MODEL SPECIES ===\n")
print(prevalence_direction)

In [ ]:
# Direction for abundance species
abundance_direction <- signi_adjust %>%
  filter(metadata == "Group",
         model == "abundance",
         feature %in% only_abund) %>%
  select(feature, coef, qval_individual) %>%
  mutate(
    direction = ifelse(coef > 0,
                       "Higher abundance in LGA",
                       "Higher abundance in AGA"),
    coef = round(coef, 3),
    qval_individual = round(qval_individual, 4)
  ) %>%
  arrange(desc(coef))

cat("=== ABUNDANCE MODEL SPECIES ===\n")
print(abundance_direction)

In [ ]:
# To identify the family, genus of the uncharacterized taxa
SGBB_class <- read_tsv("/home/anusha/sp_data/mpa_vJan25_CHOCOPhlAnSGB_202503_species.txt", col_names = c("SGB_ID", "Taxonomy"))


In [ ]:
SGBB_class_2 <- SGBB_class %>%
  separate(Taxonomy, into = c("Kingdom","Phylum","Class","Order","Family","Genus","Species"), sep = "\\|", fill = "right")


In [ ]:
SGBB_class_2 <- SGBB_class %>%
  separate(
    Taxonomy,
    into = c("Kingdom","Phylum","Class","Order","Family","Genus","Species","Extra"),
    sep = "\\|",
    fill = "right",
    extra = "merge"
  )


In [ ]:
lga_asso <- c('SGB15413', 'SGB15267', 'SGB15382', 'SGB4159', 'SGB3940', 'SGB14260', 'SGB15411','SGB15256')
aga_asso <- c('SGB4574')

In [ ]:
# Extract taxonomy 
full_taxonomy_lga <- SGBB_class_2 %>%
  filter(SGB_ID %in% lga_asso) %>%
  select(SGB_ID, Phylum, Class, Order, 
         Family, Genus, Species)

full_taxonomy_lga

In [ ]:
full_taxonomy_aga <- SGBB_class_2 %>%
  filter(SGB_ID %in% aga_asso) %>%
  select(SGB_ID, Phylum, Class, Order, 
         Family, Genus, Species)

full_taxonomy_aga